# Phase 3 — Vendor Performance: Data Loading & KPI Engineering

Goal:
1. Connect to SQL Server from Python
2. Pull dbo.VendorSummary into a pandas DataFrame
3. Clean the data (NULLs, whitespace, types)
4. Engineer KPI columns

In [3]:
%pip install sqlalchemy

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 12.1 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 8.6 MB/s  0:00:00

   ---------------------------------------- 0/2 [greenlet]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [s


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from sqlalchemy import create_engine
import urllib.parse
print("Pandas version:", pd.__version__)

Pandas version: 2.3.3


In [7]:
%pip install pyodbc

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
Server = 'DESKTOP-ASPE07O'
Database = 'Vendor'

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={Server};DATABASE={Database};Trusted_Connection=yes;"

)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
try:
    with engine.connect() as connection:
        print("Connection to the database was successful.")
except Exception as e:
    print(f"Error connecting to the database: {e}")

Connection to the database was successful.


In [9]:
query = "SELECT * FROM dbo.VendorSummary"
df = pd.read_sql(query, engine)

print("Shape:", df.shape)
df.head()

Shape: (12261, 18)


,Brand,Description,VendorNumber,VendorName,RetailPrice,ListedPurchasePrice,Classification,TotalPurchaseQunatity,TotalPurchaseDollars,AveragePurchasePrice,TotalSalesQuantity,TotalSalesDollars,TotalExciseTax,TotalFreightcost,BegInventoryQty,BegInventoryValue,EndInventoryQty,EndInventoryValue
0,31318,Ch de la Ragotiere Muscadet,1587,VINEYARD BRANDS INC,10.990000,7.580000,2,853.0,6465.739937,7.580000,765.0,8782.349983,85.330000,6070.089967,124.0,1362.759972,212.0,2965.879951
1,4225,Goslings Black Seal Bermuda,1485,CASTLE BRANDS CORP.,24.990000,18.370001,1,20089.0,369034.931326,18.370001,20179.0,522535.207651,37080.620268,8497.589942,3911.0,97735.889105,3821.0,99307.789125
2,17346,Ridge Estate Merlot,4425,MARTIGNETTI COMPANIES,47.990002,30.959999,2,61.0,1888.559967,30.959999,48.0,2303.520054,5.320000,144929.240166,34.0,1631.660057,47.0,2255.530079
3,14538,Ch St Michel Brdx Superur,4425,MARTIGNETTI COMPANIES,17.990000,12.070000,2,460.0,5552.199860,12.070000,539.0,9696.609915,59.600000,144929.240166,216.0,3885.839951,137.0,2464.629969
4,18014,Argyle Brut,4425,MARTIGNETTI COMPANIES,21.990000,13.600000,2,647.0,8799.199894,13.600000,689.0,14523.109903,76.430000,144929.240166,259.0,5177.409941,217.0,4337.829950


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12261 entries, 0 to 12260
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Brand                  12261 non-null  int64  
 1   Description            12260 non-null  object 
 2   VendorNumber           12261 non-null  int64  
 3   VendorName             12261 non-null  object 
 4   RetailPrice            12261 non-null  float64
 5   ListedPurchasePrice    12261 non-null  float64
 6   Classification         12261 non-null  int64  
 7   TotalPurchaseQunatity  10649 non-null  float64
 8   TotalPurchaseDollars   10649 non-null  float64
 9   AveragePurchasePrice   10649 non-null  float64
 10  TotalSalesQuantity     11225 non-null  float64
 11  TotalSalesDollars      11225 non-null  float64
 12  TotalExciseTax         11225 non-null  float64
 13  TotalFreightcost       12255 non-null  float64
 14  BegInventoryQty        8094 non-null   float64
 15  Be

In [11]:
df.isnull().sum()

Brand                       0
Description                 1
VendorNumber                0
VendorName                  0
RetailPrice                 0
ListedPurchasePrice         0
Classification              0
TotalPurchaseQunatity    1612
TotalPurchaseDollars     1612
AveragePurchasePrice     1612
TotalSalesQuantity       1036
TotalSalesDollars        1036
TotalExciseTax           1036
TotalFreightcost            6
BegInventoryQty          4167
BegInventoryValue        4167
EndInventoryQty          2608
EndInventoryValue        2608
dtype: int64

In [12]:
df['VendorName'] = df['VendorName'].str.strip()
df['Description'] = df['Description'].str.strip()

In [15]:
fill_zero_cols = [
    'TotalPurchaseQunatity','TotalPurchaseDollars','AveragePurchasePrice',
    'TotalSalesQuantity','TotalSalesDollars','TotalExciseTax','TotalFreightcost','BegInventoryQty','BegInventoryValue',
    'EndInventoryQty','EndInventoryValue'
]
df[fill_zero_cols] = df[fill_zero_cols].fillna(0)
df.isnull().sum()

Brand                    0
Description              1
VendorNumber             0
VendorName               0
RetailPrice              0
ListedPurchasePrice      0
Classification           0
TotalPurchaseQunatity    0
TotalPurchaseDollars     0
AveragePurchasePrice     0
TotalSalesQuantity       0
TotalSalesDollars        0
TotalExciseTax           0
TotalFreightcost         0
BegInventoryQty          0
BegInventoryValue        0
EndInventoryQty          0
EndInventoryValue        0
dtype: int64

In [16]:
df['Description'] = df['Description'].fillna('Unknown')

In [18]:
df.isnull().sum()

Brand                    0
Description              0
VendorNumber             0
VendorName               0
RetailPrice              0
ListedPurchasePrice      0
Classification           0
TotalPurchaseQunatity    0
TotalPurchaseDollars     0
AveragePurchasePrice     0
TotalSalesQuantity       0
TotalSalesDollars        0
TotalExciseTax           0
TotalFreightcost         0
BegInventoryQty          0
BegInventoryValue        0
EndInventoryQty          0
EndInventoryValue        0
dtype: int64

In [17]:
import numpy as np

In [19]:
# Gross Profit: actual revenue minus actual cost paid
df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']

# Profit Margin %: guard against division by zero (no sales -> undefined margin, not 0)
df['ProfitMargin'] = np.where(
    df['TotalSalesDollars'] > 0,
    (df['GrossProfit'] / df['TotalSalesDollars']) * 100,
    np.nan
)

# Stock Turnover: sales qty / average inventory
df['AvgInventory'] = (df['BegInventoryQty'] + df['EndInventoryQty']) / 2
df['StockTurnover'] = np.where(
    df['AvgInventory'] > 0,
    df['TotalSalesQuantity'] / df['AvgInventory'],
    np.nan
)

# Sales-to-Purchase Ratio: how many $ sold per $ spent buying
df['SalesToPurchaseRatio'] = np.where(
    df['TotalPurchaseDollars'] > 0,
    df['TotalSalesDollars'] / df['TotalPurchaseDollars'],
    np.nan
)

# Freight Cost %: freight as a share of what we spent buying from that vendor
df['FreightCostPct'] = np.where(
    df['TotalPurchaseDollars'] > 0,
    (df['TotalFreightcost'] / df['TotalPurchaseDollars']) * 100,
    np.nan
)

df[['Brand','VendorName','GrossProfit','ProfitMargin','StockTurnover','SalesToPurchaseRatio','FreightCostPct']].head(10)

,Brand,VendorName,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio,FreightCostPct
0,31318,VINEYARD BRANDS INC,2316.610046,26.378020,4.553571,1.358290,93.880825
1,4225,CASTLE BRANDS CORP.,153500.276325,29.376064,5.219607,1.415951,2.302652
2,17346,MARTIGNETTI COMPANIES,414.960087,18.014173,1.185185,1.219723,7674.060803
3,14538,MARTIGNETTI COMPANIES,4144.410055,42.740814,3.053824,1.746445,2610.303012
4,18014,MARTIGNETTI COMPANIES,5723.910009,39.412426,2.894958,1.650503,1647.072937
5,44186,MARTIGNETTI COMPANIES,5356.780247,75.838227,0.646091,4.138769,8492.030911
6,23130,MARTIGNETTI COMPANIES,3709.499958,68.831342,0.902439,3.208351,8627.973141
7,22294,MARTIGNETTI COMPANIES,1029.830141,12.999902,7.638554,1.149424,2102.861924
8,24734,VINEXTRA INC,235.680008,65.510341,4.000000,2.899420,423.065756
9,8018,VINEYARD BRANDS INC,50370.876860,35.547730,11.361899,1.551536,6.646443


In [20]:
df[['GrossProfit','ProfitMargin','StockTurnover','SalesToPurchaseRatio','FreightCostPct']].describe()

,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio,FreightCostPct
count,1.226100e+04,11225.000000,10662.000000,10648.000000,1.064800e+04
mean,1.060028e+04,-7.976720,18.929182,2.501247,2.300397e+04
std,4.334896e+04,433.784530,506.064239,8.436409,2.516367e+05
min,-5.200278e+04,-23730.639302,0.000000,0.000000,4.568383e-01
25%,8.069992e+00,18.060663,1.789474,1.155445,1.430019e+02
50%,7.384900e+02,31.873377,3.287290,1.437604,7.853809e+02
75%,6.683060e+03,44.940571,6.611751,1.665539,5.958416e+03
max,1.290668e+06,100.000000,48649.000000,352.928581,1.748517e+07
